<a href="https://colab.research.google.com/github/arshiii08/E-Commerce-Sales-Analysis-Dashboard/blob/main/e_commerce_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

In [ ]:
# =============================================================================
# 1. Load and Clean Sales Data (data.csv)
# =============================================================================

# Load the sales dataset. Adjust encoding if needed.
sales_df = pd.read_csv('data.csv', encoding='latin-1')
print("Sales Data Columns:")
print(sales_df.columns)

Sales Data Columns:
Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country'],
      dtype='object')


In [ ]:
# Remove duplicate rows
sales_df = sales_df.drop_duplicates()

In [ ]:
# Convert order_date column to datetime if it exists
if 'InvoiceDate' in sales_df.columns:
    sales_df['InvoiceDate'] = pd.to_datetime(sales_df['InvoiceDate'], errors='coerce')

# If 'total_price' does not exist, compute it from 'quantity' and 'price' if available.
if 'total_price' not in sales_df.columns:
    if 'Quantity' in sales_df.columns and 'UnitPrice' in sales_df.columns:
        sales_df['total_price'] = sales_df['Quantity'] * sales_df['UnitPrice']
        print("Computed 'total_price' from 'Quantity' and 'UnitPrice'.")
    else:
        raise KeyError("Columns for computing total_price are missing.")

Computed 'total_price' from 'Quantity' and 'UnitPrice'.


In [ ]:
# Clean the 'total_price' column if it contains messy string values (e.g., "$6.38$11.68").
def extract_first_number(s):
    numbers = re.findall(r'\d+\.\d+', s)
    if numbers:
        return float(numbers[0])
    return np.nan

if sales_df['total_price'].dtype == object:
    sales_df['total_price'] = sales_df['total_price'].apply(
        lambda x: extract_first_number(x) if isinstance(x, str) else x
    )

In [ ]:
# Convert the 'total_price' column explicitly to numeric
sales_df['total_price'] = pd.to_numeric(sales_df['total_price'], errors='coerce')

In [ ]:
# =============================================================================
# 2. Handle Missing Values, Outliers, and Normalize Data
# =============================================================================

# Fill missing numeric values with the median and categorical with 'Unknown'
numeric_cols = sales_df.select_dtypes(include=[np.number]).columns
sales_df[numeric_cols] = sales_df[numeric_cols].fillna(sales_df[numeric_cols].median())

categorical_cols = sales_df.select_dtypes(include=['object']).columns
sales_df[categorical_cols] = sales_df[categorical_cols].fillna('Unknown')


In [ ]:
# Cap outliers using the IQR method for numeric columns
for col in numeric_cols:
    Q1 = sales_df[col].quantile(0.25)
    Q3 = sales_df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    sales_df[col] = np.where(sales_df[col] < lower_bound, lower_bound, sales_df[col])
    sales_df[col] = np.where(sales_df[col] > upper_bound, upper_bound, sales_df[col])

In [ ]:
# Normalize numeric columns using Min-Max Scaling
scaler = MinMaxScaler()
sales_df[numeric_cols] = scaler.fit_transform(sales_df[numeric_cols])

print("\nSample of Sales Data after Cleaning and Normalization:")
print(sales_df.head())



Sample of Sales Data after Cleaning and Normalization:
  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER  0.513889   
1    536365     71053                  WHITE METAL LANTERN  0.513889   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER  0.569444   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE  0.513889   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.  0.513889   

          InvoiceDate  UnitPrice  CustomerID         Country  total_price  
0 2010-12-01 08:26:00   0.487847    0.926443  United Kingdom     0.586538  
1 2010-12-01 08:26:00   0.560764    0.926443  United Kingdom     0.678846  
2 2010-12-01 08:26:00   0.505208    0.926443  United Kingdom     0.709249  
3 2010-12-01 08:26:00   0.560764    0.926443  United Kingdom     0.678846  
4 2010-12-01 08:26:00   0.560764    0.926443  United Kingdom     0.678846  


In [ ]:
# =============================================================================
# 3. Aggregate Sales Metrics per Customer
# =============================================================================

# Group by customer_id to compute key metrics:
# - total_spending: Sum of total_price for each customer
# - avg_order_value: Mean of total_price for each customer
# - order_count: Count of orders (assuming each order has an 'order_id')
# - last_order_date: Most recent order_date per customer (for recency)
customer_sales = sales_df.groupby('CustomerID').agg({
    'total_price': ['sum', 'mean'],
    'InvoiceDate': 'max',
    'CustomerID': 'count'  # Ensure your dataset contains an order identifier
}).reset_index()

In [ ]:
# Flatten the MultiIndex columns
customer_sales.columns = ['CustomerID', 'total_spending', 'avg_order_value', 'last_order_date', 'order_count']


In [ ]:
# Calculate recency: days since the last order relative to the latest order in the dataset.
if 'InvoiceDate' in sales_df.columns:
    reference_date = sales_df['InvoiceDate'].max()
    customer_sales['recency'] = (reference_date - pd.to_datetime(customer_sales['last_order_date'])).dt.days
else:
    customer_sales['recency'] = np.nan

print("\nAggregated Customer Sales Metrics:")
print(customer_sales.head())



Aggregated Customer Sales Metrics:
   CustomerID  total_spending  avg_order_value     last_order_date  \
0    0.000000        1.000000         0.500000 2011-01-18 10:17:00   
1    0.000168      123.500366         0.678573 2011-12-07 15:52:00   
2    0.000337       28.178663         0.908989 2011-09-25 13:13:00   
3    0.000505       48.156136         0.659673 2011-11-21 09:51:00   
4    0.000673       11.293040         0.664296 2011-02-02 16:01:00   

   order_count  recency  
0            2      325  
1          182        1  
2           31       74  
3           73       18  
4           17      309  


In [ ]:
# =============================================================================
# 4. Load Demographic Data (survey.csv) and Merge
# =============================================================================

# Load the survey file containing customer demographics
survey_df = pd.read_csv('survey.csv', encoding='latin-1', low_memory=False)
print("\nSurvey Data Columns:")
print(survey_df.columns)
print("\nSurvey Data Sample:")
print(survey_df.head())


Survey Data Columns:
Index(['Duration (in seconds)', 'RecordedDate', 'ResponseID',
       'Q-prolific-mturk', 'q-demos-age', 'Q-demos-hispanic', 'Q-demos-race',
       'Q-demos-education', 'Q-demos-income', 'Q-demos-gender',
       'Q-sexual-orientation', 'Q-demos-state', 'Q-amazon-use-howmany',
       'Q-amazon-use-hh-size', 'Q-amazon-use-how-oft', 'Q-substance-use_1',
       'Q-substance-use_2', 'Q-substance-use_3', 'Q-personal_1',
       'Q-personal_2', 'Q-life-changes', 'Q-control', 'Q-altruism',
       'Q-bonus-05', 'Q-bonus-20', 'Q-bonus-50', 'Q-data-value-05',
       'Q-data-value-20', 'Q-data-value-50', 'Q-data-value-100',
       'Q-data-value-any', 'Q-data-value-any_1_TEXT', 'Q-sell-YOUR-data',
       'Q-sell-consumer-data', 'Q-small-biz-use', 'Q-census-use',
       'Q-research-society', 'Q-attn-check', 'showdata', 'incentive',
       'connect'],
      dtype='object')

Survey Data Sample:
   Duration (in seconds)     RecordedDate  ResponseID Q-prolific-mturk  \
0             

In [ ]:
# Ensure the survey file has a common key, e.g., ResponseID, along with demographic fields such as 'age' and 'gender'.
# Merge the aggregated sales data with the demographics using customer_id
# If your survey data uses a different customer identifier (e.g., 'customer_id'), change 'CustomerID' accordingly
customer_df = pd.merge(customer_sales, survey_df, left_on='CustomerID', right_on='ResponseID', how='left')
print("\nMerged Customer Data (Sales Metrics and Demographics):")
print(customer_df.head())


Merged Customer Data (Sales Metrics and Demographics):
   CustomerID  total_spending  avg_order_value     last_order_date  \
0    0.000000        1.000000         0.500000 2011-01-18 10:17:00   
1    0.000168      123.500366         0.678573 2011-12-07 15:52:00   
2    0.000337       28.178663         0.908989 2011-09-25 13:13:00   
3    0.000505       48.156136         0.659673 2011-11-21 09:51:00   
4    0.000673       11.293040         0.664296 2011-02-02 16:01:00   

   order_count  recency  Duration (in seconds) RecordedDate  ResponseID  \
0            2      325                    NaN          NaN         NaN   
1          182        1                    NaN          NaN         NaN   
2           31       74                    NaN          NaN         NaN   
3           73       18                    NaN          NaN         NaN   
4           17      309                    NaN          NaN         NaN   

  Q-prolific-mturk  ... Q-data-value-any_1_TEXT Q-sell-YOUR-data  \
0   

In [ ]:
# =============================================================================
# 5. Exploratory Data Analysis (Optional)
# =============================================================================

# Example: Plot daily sales trends
if 'order_date' in sales_df.columns:
    sales_df['order_day'] = pd.to_datetime(sales_df['order_date']).dt.date
    daily_sales = sales_df.groupby('order_day')['total_price'].sum().reset_index()

    plt.figure(figsize=(10, 6))
    plt.plot(daily_sales['order_day'], daily_sales['total_price'], marker='o')
    plt.xlabel('Date')
    plt.ylabel('Daily Sales (Normalized)')
    plt.title('Daily Sales Trends')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


In [ ]:
# =============================================================================
# 6. Customer Segmentation using K-Means Clustering
# =============================================================================

# Before clustering, we need to encode any remaining categorical variables (for demographics).
# For example, encode 'gender' (if available) as numeric.
if 'gender' in customer_df.columns:
    customer_df['gender_encoded'] = customer_df['gender'].str.lower().map({
        'male': 0,
        'female': 1
    }).fillna(2)
else:
    customer_df['gender_encoded'] = 2

# Choose features for segmentation. You can adjust these based on your needs.
# Here we use sales metrics (total_spending, avg_order_value, order_count, recency) and demographics (age, encoded gender).
features = customer_df[['total_spending', 'avg_order_value', 'Quantity', 'recency', 'q-demos-age', 'Q-demos-gender']].copy()
features.fillna(features.median(), inplace=True)  # Fill in any missing values with median

# Standardize the features so that clustering is not biased by the scale of any one variable.
std_scaler = StandardScaler()
features_scaled = std_scaler.fit_transform(features)

# Use the Elbow Method to determine the optimal number of clusters
sse = []
for k in range(1, 10):
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(features_scaled)
    sse.append(kmeans.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(range(1, 10), sse, marker='o')
plt.xlabel('Number of clusters')
plt.ylabel('Sum of Squared Errors (SSE)')
plt.title('Elbow Method to Determine Optimal Clusters')
plt.show()

# Choose an appropriate number of clusters (e.g., 3) based on the elbow plot.
optimal_k = 3
kmeans = KMeans(n_clusters=optimal_k, random_state=42)
customer_df['cluster'] = kmeans.fit_predict(features_scaled)

print("\nCustomer Segmentation Results (First 5 rows):")
print(customer_df.head())

KeyError: "['Quantity'] not in index"